# Notebook 02 — Génération des Q&R (3–4 datasets) — Option B

**LLM de génération** : **Anthropic** (`claude-sonnet-4-6`) — génération indépendante du modèle d’inférence Groq évalué (réduit le biais circulaire).

**Inputs** :
- `BASE_PATH/data/raw/wikipedia_technique.json`
- `BASE_PATH/data/raw/hal.json`
- `BASE_PATH/data/raw/lemonde.json`
- `BASE_PATH/data/raw/code_route.json` (optionnel — segments PDF *code de la route* ; ≥20 segments pour 100 paires juridiques)

**Outputs** (splits figés — Option B) :

| Split | Wikipedia | HAL FR | Le Monde | Code route (si PDF) | Total |
|-------|-----------|--------|----------|----------------------|-------|
| train | 400 ou **320** | 200 s. + 200 c. | 400 | **0** ou **80** | **1200** |
| test  | 100 ou **80** | 50 s. + 50 c. | 100 | **0** ou **20** | **300** |

**Runtime** : CPU — variable selon quotas Groq (~30–90 min selon débits)

## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation des dépendances

In [ ]:
# Installation du SDK Anthropic + json-repair pour tolérer le JSON malformé du LLM
!pip install -q anthropic json-repair

## 2. Imports, configuration et clé API

In [ ]:
# Imports des bibliothèques nécessaires
import os
import json
import time
import random
import getpass
from anthropic import Anthropic
from tqdm.notebook import tqdm

# Chemins des répertoires
RAW_PATH       = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

print(f"Répertoire source  : {RAW_PATH}")
print(f"Répertoire cible   : {PROCESSED_PATH}")

In [ ]:
# Saisie sécurisée de la clé Anthropic (https://console.anthropic.com/settings/keys)
# La clé commence en général par sk-ant-...
api_key = getpass.getpass("Entre ta clé Anthropic API : ").strip()
if not api_key.startswith("sk-ant-"):
    raise ValueError("Clé API Anthropic invalide : elle doit commencer par sk-ant-. Vérifie le copier-coller.")
anthropic_client = Anthropic(api_key=api_key)
print("Client Anthropic initialisé.")
GEN_MODEL = "claude-sonnet-4-6"
print(f"Modèle génération dataset : {GEN_MODEL}")

## 3. Chargement des données brutes depuis Drive

In [ ]:
def load_json(path, label=''):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé ({label}) : {len(data)} docs")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Introuvable : {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des datasets bruts...")
wiki_docs    = load_json(os.path.join(RAW_PATH, 'wikipedia_technique.json'), 'Wikipedia technique')
hal_docs     = load_json(os.path.join(RAW_PATH, 'hal.json'),                 'HAL multisauts (FR)')
lemonde_docs = load_json(os.path.join(RAW_PATH, 'lemonde.json'),             'Le Monde temporel')
legal_docs   = load_json(os.path.join(RAW_PATH, 'code_route.json'),          'Code de la route (PDF)')
HAS_LEGAL    = len(legal_docs) > 0
print(f"\nTotal : {len(wiki_docs)+len(hal_docs)+len(lemonde_docs)+len(legal_docs)} documents (juridique : {len(legal_docs)})")

# ── Inférence de recency_category depuis le champ 'date' ─────────────────────
def infer_recency(date_str):
    """Déduit la fraîcheur d'un document depuis sa date ISO (YYYY-MM-DD)."""
    try:
        year = int(str(date_str)[:4])
        if year >= 2024: return 'récent'
        if year >= 2022: return 'intermédiaire'
        return 'fondamental'
    except Exception:
        return 'inconnu'

for doc in wiki_docs + hal_docs + lemonde_docs + legal_docs:
    doc['recency_category'] = infer_recency(doc.get('date', ''))

from collections import Counter
rc = Counter(d['recency_category'] for d in wiki_docs + hal_docs + lemonde_docs + legal_docs)
print(f"  Recency distribution : {dict(rc)}")

## 4. Génération des paires Q&R via Anthropic API (Sonnet 4.6)

In [ ]:
# ── Prompt standard (Wikipedia + Le Monde) ───────────────────────────────────
STANDARD_PROMPT = """Tu es un expert. À partir du texte ci-dessous, génère EXACTEMENT 5 paires question-réponse EN FRANÇAIS.

Génère dans cet ordre :
1. Question FACTUELLE (fait précis, date, chiffre, entité nommée)
2. Question FACTUELLE (idem)
3. Question de SYNTHÈSE (compare ou résume plusieurs concepts)
4. Question de SYNTHÈSE (idem)
5. Question de COMPRÉHENSION (causalité, implication, pourquoi/comment)

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du texte (20-150 mots) qui contient/justifie la réponse.
INTERDIT d'écrire "Texte source", "référence au texte" ou le titre seul.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"..."}}
Types autorisés : "factuel", "synthese", "comprehension"

Exemple :
[{{"question":"En quelle année X a été fondé ?","answer":"2020","context":"X a été fondé en 2020 par des chercheurs issus de Google Brain, avec pour objectif de...","type":"factuel"}}]

Texte source :
{content}"""

# ── Prompt HAL simples (3 questions par papier — 100% français) ──────────────
HAL_SIMPLE_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique EN FRANÇAIS, génère EXACTEMENT 3 questions factuelles simples EN FRANÇAIS.

Chaque question porte sur UN SEUL fait (méthode utilisée, métrique obtenue, dataset employé, contribution principale).
Chaque réponse tient en 1-2 phrases courtes.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du résumé (20-100 mots) qui contient la réponse.
INTERDIT d'écrire "Texte source" ou similaire.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"simple"}}

Exemple :
[{{"question":"Quel dataset est utilisé pour l'évaluation ?","answer":"MMLU","context":"We evaluate our model on the MMLU benchmark, achieving state-of-the-art performance across 57 tasks.","type":"simple"}}]

Résumé :
{content}"""

# ── Prompt HAL complexes / multi-sauts (2 questions par papier) ─────────────
HAL_COMPLEX_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique EN FRANÇAIS, génère EXACTEMENT 2 questions complexes EN FRANÇAIS.

Ces questions nécessitent de RELIER PLUSIEURS INFORMATIONS du texte pour répondre (multi-sauts de raisonnement).
Par exemple : "Pourquoi la méthode X obtient-elle de meilleurs résultats que Y sur Z ?"
Chaque réponse fait 3-4 phrases et synthétise plusieurs éléments du résumé.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT 1-2 extraits du résumé (30-200 mots) qui, ensemble, permettent de répondre.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"complexe"}}

Résumé :
{content}"""
THROTTLE_S  = 0.6    # Anthropic — petite marge anti-rate-limit
MAX_CONTENT = 4000   # chars envoyés au LLM

In [ ]:
import re as _re
from json_repair import repair_json

VALID_TYPES = {"factuel", "synthese", "comprehension", "simple", "complexe"}

def _extract_and_repair(raw_text):
    """Extrait le JSON d'une réponse LLM et le répare si nécessaire."""
    # 1. Cherche un tableau JSON entre crochets
    match = _re.search(r'(\[.*\])', raw_text, _re.DOTALL)
    if match:
        candidate = match.group(1)
    else:
        candidate = raw_text.strip()
    # 2. Tente de réparer le JSON malformé
    return repair_json(candidate, return_objects=False)

def _call_anthropic(prompt, retries=4):
    """Appelle l'API Anthropic (messages) et retourne le texte brut."""
    for attempt in range(retries):
        try:
            resp = anthropic_client.messages.create(
                model=GEN_MODEL,
                max_tokens=2048,
                temperature=0.2,
                messages=[{"role": "user", "content": prompt}],
            )
            chunks = []
            for b in getattr(resp, "content", []) or []:
                txt = getattr(b, "text", None)
                if txt:
                    chunks.append(txt)
            return "\n".join(chunks).strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = 5 * (2 ** attempt)
                print(f"    [QUOTA] Attente {wait}s...")
                time.sleep(wait)
            else:
                print(f"    [ERROR] {err[:120]}")
                time.sleep(2)
    return ""

def _parse_pairs(raw_text, document, forced_type=None):
    """Extrait et valide les paires Q&R depuis la réponse LLM."""
    content = document.get('content', '')
    json_str = _extract_and_repair(raw_text)
    pairs = json.loads(json_str)
    if not isinstance(pairs, list):
        return []
    validated = []
    for pair in pairs:
        if not (isinstance(pair, dict) and pair.get('question') and pair.get('answer')):
            continue
        q_type = forced_type or str(pair.get('type', 'factuel')).lower().strip()
        if q_type not in VALID_TYPES:
            q_type = forced_type or 'factuel'
        ctx = str(pair.get('context', '')).strip()
        if len(ctx) < 30:
            ctx = content[:400]
        validated.append({
            "question":      str(pair['question']).strip(),
            "answer":        str(pair['answer']).strip(),
            "context":       ctx,
            "source_id":     document.get('id', ''),
            "source":            document.get('source', ''),
            "langue":            "fr",
            "title":             document.get('title', ''),
            "date":              document.get('date', ''),
            "dataset_type":      document.get('dataset_type', ''),
            "question_type":     q_type,
            "recency_category": document.get('recency_category', 'inconnu'),
        })
    return validated


def generate_standard_qa(document, retries=4):
    """Génère 5 paires standard (Wikipedia / Le Monde)."""
    content = document.get('content', '')
    if len(content) < 100:
        return []
    prompt = STANDARD_PROMPT.format(content=content[:MAX_CONTENT])
    for attempt in range(retries):
        try:
            raw = _call_anthropic(prompt)
            if not raw:
                continue
            pairs = _parse_pairs(raw, document)
            if pairs:
                return pairs
        except (json.JSONDecodeError, ValueError) as e:
            print(f"    [WARN] {attempt+1}/{retries} JSON invalide '{document.get('title','')[:30]}': {str(e)[:50]}")
            time.sleep(2 ** attempt)
    return []


def generate_arxiv_qa(document, retries=4):
    """Génère 3 questions simples + 2 questions complexes pour un papier Arxiv."""
    content = document.get('content', '')
    if len(content) < 100:
        return []
    all_pairs = []
    for prompt_template, forced_type, n_expected in [
                (HAL_SIMPLE_PROMPT,  'simple',   3),
        (HAL_COMPLEX_PROMPT, 'complexe', 2),
    ]:
        prompt = prompt_template.format(content=content[:MAX_CONTENT])
        for attempt in range(retries):
            try:
                raw = _call_anthropic(prompt)
                if not raw:
                    break
                pairs = _parse_pairs(raw, document, forced_type=forced_type)
                if pairs and len(pairs) >= n_expected:
                    all_pairs.extend(pairs[:n_expected])
                    break
            except (json.JSONDecodeError, ValueError) as e:
                print(f"    [WARN] {attempt+1}/{retries} Arxiv {forced_type} '{document.get('title','')[:30]}': {str(e)[:50]}")
                time.sleep(2 ** attempt)
        time.sleep(THROTTLE_S)
    return all_pairs

In [ ]:
print("=" * 60)
print("Génération des Q&R — datasets (Option B + juridique optionnel)")
print("=" * 60)

# Quotas : len(docs)×5 (l'idéal Option B = 100 docs → 500 paires ; moins d'articles → cible plus basse)
MIN_PAIRS_PER_DOC = 5
MAX_DOC_ROUNDS    = 18   # relances par article si JSON partiel / < 5 paires
TARGET_WIKI       = len(wiki_docs) * MIN_PAIRS_PER_DOC
TARGET_HAL        = len(hal_docs) * MIN_PAIRS_PER_DOC
TARGET_LEMONDE    = len(lemonde_docs) * MIN_PAIRS_PER_DOC
TARGET_LEGAL      = (min(20, len(legal_docs)) * MIN_PAIRS_PER_DOC) if HAS_LEGAL else 0
import random

def _generate_standard_until_quota(doc, label=""):
    """Accumule des paires à questions distinctes sur plusieurs tours (évite rester bloqué à 1/5)."""
    seen, best = set(), []
    for r in range(MAX_DOC_ROUNDS):
        batch = generate_standard_qa(doc)
        for p in batch:
            k = str(p.get('question', '')).strip().lower()[:160]
            if not k or k in seen:
                continue
            seen.add(k)
            best.append(p)
            if len(best) >= MIN_PAIRS_PER_DOC:
                return best[:MIN_PAIRS_PER_DOC]
        time.sleep(THROTTLE_S + min(r, 6))
    if len(best) < MIN_PAIRS_PER_DOC:
        print(f"  [WARN] {label} '{str(doc.get('title',''))[:50]}' : {len(best)}/{MIN_PAIRS_PER_DOC} paires")
    return best[:MIN_PAIRS_PER_DOC] if len(best) >= MIN_PAIRS_PER_DOC else best

def _generate_hal_until_quota(doc):
    """Accumule 3 simples + 2 complexes (questions distinctes par type)."""
    seen_s, seen_c = set(), set()
    simple, complexe = [], []
    for r in range(MAX_DOC_ROUNDS):
        batch = generate_arxiv_qa(doc)
        for p in batch:
            k = str(p.get('question', '')).strip().lower()[:160]
            if not k:
                continue
            qt = str(p.get('question_type', 'simple')).lower()
            if qt == 'complexe':
                if k in seen_c:
                    continue
                seen_c.add(k)
                complexe.append(p)
            else:
                if k in seen_s:
                    continue
                seen_s.add(k)
                simple.append(p)
        if len(simple) >= 3 and len(complexe) >= 2:
            return simple[:3] + complexe[:2]
        time.sleep(THROTTLE_S + min(r, 6))
    merged = simple[:3] + complexe[:2]
    if len(merged) < MIN_PAIRS_PER_DOC:
        print(f"  [WARN] HAL '{str(doc.get('title',''))[:50]}' : {len(merged)}/{MIN_PAIRS_PER_DOC} paires")
    return merged[:MIN_PAIRS_PER_DOC] if len(merged) >= MIN_PAIRS_PER_DOC else merged

# ── Wikipedia (standard) ─────────────────────────────────────────────────────
print("\n[1/3] Wikipedia technique...")
wiki_qa = []
for doc in tqdm(wiki_docs, desc="Wikipedia Q&R"):
    pairs = _generate_standard_until_quota(doc, "Wiki")
    for i, p in enumerate(pairs):
        p['pair_id'] = f"wiki_qa_{len(wiki_qa):04d}"
        wiki_qa.append(p)
    time.sleep(THROTTLE_S)
print(f"  => {len(wiki_qa)} paires générées depuis {len(wiki_docs)} articles (cible {TARGET_WIKI})")

# ── HAL (simple + complexe) — 100% français ──────────────────────────────────
print("\n[2/3] HAL multisauts (FR)...")
hal_qa = []
for doc in tqdm(hal_docs, desc="HAL Q&R"):
    pairs = _generate_hal_until_quota(doc)
    for p in pairs:
        p['pair_id'] = f"hal_qa_{len(hal_qa):04d}"
        hal_qa.append(p)
hal_simple  = [p for p in hal_qa if p['question_type'] == 'simple']
hal_complex = [p for p in hal_qa if p['question_type'] == 'complexe']
print(f"  => {len(hal_qa)} paires  ({len(hal_simple)} simples + {len(hal_complex)} complexes) — cible {TARGET_HAL}")

# ── Le Monde (standard) ──────────────────────────────────────────────────────
print("\n[3/3] Le Monde temporel...")
lemonde_qa = []
for doc in tqdm(lemonde_docs, desc="Le Monde Q&R"):
    pairs = _generate_standard_until_quota(doc, "News")
    for p in pairs:
        p['pair_id'] = f"lemonde_qa_{len(lemonde_qa):04d}"
        lemonde_qa.append(p)
    time.sleep(THROTTLE_S)
print(f"  => {len(lemonde_qa)} paires générées depuis {len(lemonde_docs)} articles (cible {TARGET_LEMONDE})")

# ── Code de la route (juridique) : jusqu'à 20 segments tirés au hasard (seed 42) ─
legal_qa = []
if HAS_LEGAL:
    if len(legal_docs) < 20:
        raise RuntimeError(
            "code_route.json : au moins 20 segments sont nécessaires pour 100 paires juridiques (5 paires/segment). "
            "Augmentez max_chunks ou la taille du PDF dans 01_scraping, puis regénérez code_route.json."
        )
    random.seed(42)
    _s = legal_docs[:]
    random.shuffle(_s)
    legal_sample = _s[:20]
    print(f"\n[4/4] Code de la route (juridique) — {len(legal_sample)} segments...")
    for doc in tqdm(legal_sample, desc="Code route Q&R"):
        pairs = _generate_standard_until_quota(doc, "CDR")
        for p in pairs:
            p['pair_id'] = f"legal_qa_{len(legal_qa):04d}"
            legal_qa.append(p)
        time.sleep(THROTTLE_S)
    print(f"  => {len(legal_qa)} paires générées (cible {TARGET_LEGAL})")

# ── Contrôle quotas globaux (sinon split 1200/300 impossible) ─────────────────
_quota = [("Wikipedia", len(wiki_qa), TARGET_WIKI), ("HAL", len(hal_qa), TARGET_HAL), ("Le Monde", len(lemonde_qa), TARGET_LEMONDE)]
if HAS_LEGAL:
    _quota.append(("Code route", len(legal_qa), TARGET_LEGAL))
for name, got, tgt in _quota:
    if got < tgt:
        raise RuntimeError(
            f"Quota insuffisant — {name}: {got}/{tgt} paires. "
            f"Vérifiez 01_scraping (100 articles/source ou PDF juridique), longueur du texte (>100 car.), puis relancez cette section."
        )
print("\n  [OK] Quotas atteints — split Option B possible.")

## 5. Mélange et division train / test

Split **stratifié** sur `recency_category` (même proportions train/test que dans le pool de chaque source, seed 42), en plus des quotas par source (Wiki / HAL simple / HAL complexe / Le Monde / optionnellement Code route si `code_route.json` non vide).

In [ ]:
import random
import math
random.seed(42)

def exact_split(pairs, n_train, n_test, label='', stratify_key='recency_category'):
    """Sélectionne n_train + n_test paires (pool mélangé puis tronqué si surplus).\n
    Si pool < n_train+n_test (ex. HAL complexe max 200 pour 250 demandés), quotas réduits proportionnellement pour éviter un test vide. Stratification si plusieurs strates."""
    total0 = len(pairs)
    if total0 == 0:
        return [], []
    pool = pairs[:]
    random.shuffle(pool)
    T_req = n_train + n_test
    if total0 > T_req:
        pool = pool[:T_req]
    total = len(pool)
    n_train_o, n_test_o = n_train, n_test
    T = n_train + n_test
    if total < T:
        print(f"  [WARN] {label}: pool={total} < {n_train_o}+{n_test_o} → split proportionnel (évite test vide).")
        ratio = (n_train / T) if T > 0 else 0.5
        n_train = max(0, min(total, int(round(total * ratio))))
        n_test = total - n_train
        if n_test_o > 0 and n_test == 0 and total > 1:
            n_test = min(n_test_o, max(1, total // 5))
            n_train = total - n_test
        T = n_train + n_test
    if not stratify_key:
        return pool[:n_train], pool[n_train:T]
    cats = sorted({p.get(stratify_key, 'inconnu') for p in pool})
    if len(cats) <= 1:
        return pool[:n_train], pool[n_train:T]
    by_cat = {c: [p for p in pool if p.get(stratify_key, 'inconnu') == c] for c in cats}
    for c in cats:
        random.shuffle(by_cat[c])
    sizes = {c: len(by_cat[c]) for c in cats}
    tot = sum(sizes.values())
    floor_t = {c: int(math.floor(sizes[c] * n_train / tot)) for c in cats}
    rem = n_train - sum(floor_t.values())
    order = sorted(cats, key=lambda c: (sizes[c] * n_train / tot - floor_t[c]), reverse=True)
    alloc = dict(floor_t)
    for i in range(rem):
        alloc[order[i % len(order)]] += 1
    train, test = [], []
    for c in cats:
        lst = by_cat[c]
        k = min(alloc.get(c, 0), len(lst))
        train.extend(lst[:k])
        test.extend(lst[k:])
    random.shuffle(train)
    random.shuffle(test)
    return train, test

# ── Option B 80/20 : 1200 train + 300 test ──────────────────────────────────
# Sans juridique : Wiki 400+100, HAL 200+50 / 200+50, Le Monde 400+100
# Avec juridique : Wiki 320+80, HAL idem, Le Monde 400+100, Code route 80+20

if HAS_LEGAL:
    wiki_train, wiki_test = exact_split(wiki_qa, 320, 80, 'Wikipedia')
    legal_train, legal_test = exact_split(legal_qa, 80, 20, 'Code route')
else:
    wiki_train, wiki_test = exact_split(wiki_qa, 400, 100, 'Wikipedia')
    legal_train, legal_test = [], []

# ── HAL : équilibre simple/complexe ──────────────────────────────────────────
hal_s_train, hal_s_test = exact_split(hal_simple,  200, 50, 'HAL simple')
hal_c_train, hal_c_test = exact_split(hal_complex, 200, 50, 'HAL complexe')
hal_train = hal_s_train + hal_c_train
hal_test  = hal_s_test  + hal_c_test
print(f"  [CHECK] HAL test : {len(hal_s_test)} simple + {len(hal_c_test)} complexe | Wiki test (technique) : {sum(1 for p in wiki_test if p.get('dataset_type')=='technique')}")

# ── Le Monde : 400 train + 100 test ──────────────────────────────────────────
lemonde_train, lemonde_test = exact_split(lemonde_qa, 400, 100, 'Le Monde')

# ── Assemblage final ─────────────────────────────────────────────────────────
train_data = wiki_train + hal_train + lemonde_train + legal_train
test_data  = wiki_test  + hal_test  + lemonde_test + legal_test
random.shuffle(train_data)
random.shuffle(test_data)

# Re-indexation des pair_id
for i, p in enumerate(train_data): p['pair_id'] = f"train_{i:04d}"
for i, p in enumerate(test_data):  p['pair_id'] = f"test_{i:04d}"

import collections
print("\n" + "=" * 55)
print("RÉPARTITION FINALE DU DATASET")
print("=" * 55)
for split_name, split in [("TRAIN (1200)", train_data), ("TEST  (300)", test_data)]:
    print(f"\n  {split_name}")
    by_ds = collections.Counter(p['dataset_type']   for p in split)
    by_qt = collections.Counter(p['question_type']  for p in split)
    by_rc = collections.Counter(p.get('recency_category', 'inconnu') for p in split)
    print(f"    Par source     : {dict(by_ds)}")
    print(f"    Par type Q     : {dict(by_qt)}")
    print(f"    Par temporalité: {dict(by_rc)}")
print(f"\n  Totaux : {len(train_data)} train + {len(test_data)} test = {len(train_data)+len(test_data)} paires")
assert len(train_data) <= 1200 and len(test_data) <= 300, \
    f'[WARN] Taille inattendue : {len(train_data)} train / {len(test_data)} test (vérifiez le nombre de docs scrappés)'

## 6. Statistiques du dataset

In [ ]:
# Statistiques détaillées du dataset train + test
import statistics, re

def field_stats(pairs, field):
    counts = [len(str(p.get(field, '')).split()) for p in pairs]
    if not counts:
        return None
    return {"min": min(counts), "mean": round(statistics.mean(counts),1),
            "median": statistics.median(counts), "max": max(counts), "total": sum(counts)}

def print_split_stats(split_name, pairs):
    print(f"\n  ── {split_name} ({len(pairs)} paires) ──")
    print(f"  {'Champ':<12} {'Min':>5} {'Moy':>7} {'Médiane':>8} {'Max':>5} {'Total mots':>12}")
    print(f"  {'─'*12} {'─'*5} {'─'*7} {'─'*8} {'─'*5} {'─'*12}")
    for field, label in [("question","Question"), ("answer","Réponse"), ("context","Contexte")]:
        s = field_stats(pairs, field)
        if s:
            print(f"  {label:<12} {s['min']:>5} {s['mean']:>7} {s['median']:>8.0f} {s['max']:>5} {s['total']:>12,}")

    # Types de questions
    type_dist = {}
    for p in pairs:
        t = p.get('question_type', 'inconnu')
        type_dist[t] = type_dist.get(t, 0) + 1
    print(f"\n  Types de questions :")
    for t, cnt in sorted(type_dist.items(), key=lambda x: -x[1]):
        bar = "█" * (cnt // max(1, len(pairs) // 30))
        print(f"    {t:<16} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)  {bar}")

    # Strates temporelles
    rc_dist = {}
    for p in pairs:
        rc = p.get('recency_category', 'inconnu')
        rc_dist[rc] = rc_dist.get(rc, 0) + 1
    print(f"\n  Strates temporelles :")
    for cat in ["récent", "intermédiaire", "fondamental", "inconnu"]:
        cnt = rc_dist.get(cat, 0)
        if cnt > 0:
            print(f"    {cat:<16} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)")

    # Sources
    src_dist = {}
    for p in pairs:
        src_dist[p.get('source','?')] = src_dist.get(p.get('source','?'), 0) + 1
    print(f"\n  Sources :")
    for src, cnt in sorted(src_dist.items(), key=lambda x: -x[1]):
        print(f"    {src:<20} : {cnt:>4} ({100*cnt/len(pairs):.1f}%)")

print("=" * 65)
print("STATISTIQUES DU DATASET (langue : français)")
print("=" * 65)
print_split_stats("TRAIN", train_data)
print_split_stats("TEST ", test_data)

all_qa_pairs = wiki_qa + hal_qa + lemonde_qa + legal_qa
all_text = " ".join(p.get('question','') + " " + p.get('answer','') for p in all_qa_pairs)
vocab = set(re.sub(r'[^a-zàâéèêëîïôùûüç\s]', '', all_text.lower()).split())
print(f"\n  Vocabulaire unique (approx.) : {len(vocab):,} tokens")
print(f"  Total paires générées        : {len(all_qa_pairs)}")
print(f"  Paires conservées            : {len(train_data)+len(test_data)} (train {len(train_data)} + test {len(test_data)})")
print("=" * 65)
print("\n✔ Notebook 02 terminé. Lancez 03_baseline_rag.ipynb pour la suite.")

## 6. Sauvegarde sur Drive

In [ ]:
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_path = os.path.join(PROCESSED_PATH, 'train.json')
test_path  = os.path.join(PROCESSED_PATH, 'test.json')

for data, path, label in [(train_data, train_path, 'train'), (test_data, test_path, 'test')]:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"{label}.json : {len(data)} paires  ({os.path.getsize(path)/1024:.1f} Ko)  → {path}")

## 7. Résumé final

In [ ]:
import os
train_path = os.path.join(BASE_PATH, 'data', 'processed', 'train.json')
test_path  = os.path.join(BASE_PATH, 'data', 'processed', 'test.json')

print("=" * 60)
print("RÉSUMÉ FINAL — Notebook 02 Dataset Builder (Anthropic)")
print("=" * 60)
print(f"\n  Données brutes chargées :")
print(f"    Wikipedia  : {len(wiki_docs)} articles")
print(f"    HAL        : {len(hal_docs)} papiers")
print(f"    Le Monde   : {len(lemonde_docs)} articles")
print(f"    Code route : {len(legal_docs)} segments (PDF)")
print(f"\n  Paires Q&R générées :")
print(f"    Wikipedia  : {len(wiki_qa)} paires")
print(f"    HAL        : {len(hal_qa)} paires  ({len(hal_simple)} simples + {len(hal_complex)} complexes)")
print(f"    Le Monde   : {len(lemonde_qa)} paires")
print(f"    Juridique  : {len(legal_qa)} paires")
print(f"    Total      : {len(wiki_qa)+len(hal_qa)+len(lemonde_qa)+len(legal_qa)} paires générées")
print(f"\n  Splits sauvegardés :")
print(f"    train.json : {len(train_data)} paires  → {train_path}")
print(f"    test.json  : {len(test_data)} paires  → {test_path}")
print(f"    Total      : {len(train_data)+len(test_data)} paires conservées")
print(f"\n  Fichiers sur Drive :")
for p in [train_path, test_path]:
    if os.path.exists(p):
        print(f"    ✓ {os.path.basename(p)}  ({os.path.getsize(p)/1024:.1f} Ko)")
    else:
        print(f"    ✗ {os.path.basename(p)}  INTROUVABLE")
print("\n→ Lancez 03_baseline_rag.ipynb pour l'étape suivante.")
print("=" * 60)
